# DrugReview: evaluation of the supervision ladder

This notebook reproduces the model-evaluation results of the manuscript from the saved outputs of the selected models. It trains nothing: it reads the test predictions, validation histories, and fusion weights written by the rung notebooks (`01`–`07` and `ablation/A0`) and scores them. Every result is printed below and written to `08_Manuscript_evaluation/outputs/`.

| Section | Result | Manuscript |
|---|---|---|
| 1 | Rungs 1 and 2, text only and with metadata; paired metadata differences; ALBERT minus BioBERT | Table 5, Table S6, Section 4.2.1 |
| 2 | Rung 3, the aspect baseline | Table 7 |
| 3 | Rungs 4–7, holistic and aspect scores | Table 8 |
| 4 | Paired holistic contrasts and the equivalence margin | Table 9, Table S9 |
| 5 | Aspect agreement relative to rung 3 | Table 10 |
| 6 | Checkpoint selection on aspect agreement | Table S10 |
| 7 | Fusion weights by holistic class | Table S12 |

Intervals are 95% percentile intervals from a bootstrap of narrative groups on the test partition (reviews that share a narrative are resampled together; models scored on the same draws are paired): 1,000 draws with seed 2025, and 10,000 draws for the paired contrasts. The rung 1–2 summaries in section 1 are read from those notebooks, which use the same resampling with seed 20260919.

## 0. Setup: paths, constants, bootstrap

In [1]:
import os, json, glob, warnings
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, cohen_kappa_score
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 200)

# ROOT is the folder that holds 00_labelling/, 00_split/, 01_OriginalLabel/, ... (Drive: NLP_Projects/03_DrugReview; local: "01. Colab backup").
if os.path.isdir("/content") and not os.environ.get("DRUGREVIEW_ROOT"):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)

def resolve_project_root():
    override = os.environ.get("DRUGREVIEW_ROOT")
    if override:
        candidate = Path(override).expanduser().resolve()
        if not candidate.is_dir():
            raise FileNotFoundError(f"DRUGREVIEW_ROOT does not exist: {candidate}")
        return candidate
    for base in [Path.cwd(), *Path.cwd().parents, Path("/content/drive/MyDrive/NLP_Projects/03_DrugReview")]:
        for candidate in (base, base / "DrugReview_new"):
            if (candidate / "01_OriginalLabel").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Set DRUGREVIEW_ROOT to the folder containing 01_OriginalLabel, 00_split, and the run data.")

ROOT = resolve_project_root()
OUT = Path(os.environ.get("DRUGREVIEW_EVAL_OUT", ROOT / "08_Manuscript_evaluation" / "outputs")); OUT.mkdir(parents=True, exist_ok=True)

PROTOCOL_ID = "v2_ef356fdf87"
REL = ROOT / "00_labelling" / "output_v2" / PROTOCOL_ID
CLASSES = ["VERY_NEGATIVE", "NEGATIVE", "NEUTRAL", "POSITIVE", "VERY_POSITIVE"]
A3 = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
ASPECTS = ["efficacy", "safety", "burden", "cost"]
ENCODERS = ("albert", "biobert")
RUNS = {"03": "03_Full_aspects/outputs/aspect_auxiliary_v2_3class",
        "04": "04_Holistic_all_aspects/outputs/aspect_auxiliary_v2_3class",
        "A0": "ablation/A0_Matched_holistic_control/outputs/aspect_auxiliary_v2_3class",
        "5p": "05_Confirmatory/outputs/multilevel_representation_only_v2",
        "5r": "05_Confirmatory/outputs/multilevel_representation_residual_v2",
        "5j": "05_Confirmatory/outputs/multilevel_probability_bottleneck_v2",
        "5d": "05_Confirmatory/outputs/multilevel_detached_probability_bottleneck_v2",
        "6ff": "06_Semi_confirmatory/outputs/semi_4anchored_4free_v2",
        "6a": "06_Semi_confirmatory/outputs/semi_4anchored_0free_v2",
        "6f": "06_Semi_confirmatory/outputs/semi_0anchored_4free_v2",
        "7e": "07_Evidence_hierarchical/outputs/evidence_hierarchical_v2_evidence_1",
        "7c": "07_Evidence_hierarchical/outputs/evidence_hierarchical_v2_evidence_0"}
NAMES = {"03": "3 aspect baseline", "04": "4 parallel", "A0": "4 no aspect heads", "5p": "5 fusion", "5r": "5 residual",
         "5j": "5 bottleneck, joint", "5d": "5 bottleneck, detached", "6ff": "6 4 anchored + 4 free", "6a": "6 4 anchored + 0 free",
         "6f": "6 0 anchored + 4 free", "7e": "7 evidence loss on", "7c": "7 evidence loss off"}

def test_predictions(run, enc, usecols=None):
    df = pd.read_csv(ROOT / RUNS[run] / enc / f"{enc}_test_predictions.csv", usecols=usecols)
    return df.sort_values("uniqueID").reset_index(drop=True)

# Narrative-group bootstrap on group-level confusion matrices: each draw resamples the narrative groups of the test partition
# with replacement (numpy default_rng(seed).integers(0, G, G)), so reviews that share a narrative stay together, and every
# model scored on the same draws is paired. 95% percentile intervals.
def confusion_metrics(M):
    """M: (..., K, K) counts, rows = true. Returns weighted F1, macro F1, quadratic-weighted kappa."""
    K = M.shape[-1]; W = np.array([[(i - j) ** 2 for j in range(K)] for i in range(K)], float)
    tp = np.einsum("...ii->...i", M); rows = M.sum(-1); cols = M.sum(-2); n = rows.sum(-1, keepdims=True)
    den = 2 * tp + (cols - tp) + (rows - tp); f1 = np.where(den > 0, 2 * tp / np.where(den > 0, den, 1), 0.0)
    wf1 = (f1 * rows).sum(-1) / n[..., 0]; mf1 = f1.mean(-1)
    E = rows[..., :, None] * cols[..., None, :] / n[..., None]; qwk = 1 - (W * M).sum((-1, -2)) / (W * E).sum((-1, -2))
    return wf1, mf1, qwk

def group_draws(groups, n, seed):
    codes = pd.factorize(pd.Series(groups))[0]; G = int(codes.max() + 1); rng = np.random.default_rng(seed)
    counts = np.empty((n, G))                              # filled in place: 10,000 x ~19,000 draws stay near 1.5 GB
    for i in range(n):
        counts[i] = np.bincount(rng.integers(0, G, G), minlength=G)
    return codes, G, counts

def boot_metrics(y, preds, groups, K, n=1000, seed=2025):
    """preds: dict name -> predicted ids (aligned with y). Returns dict name -> (point (wf1, mf1, qwk), draws (n, 3))."""
    codes, G, cnt = group_draws(groups, n, seed); y = np.asarray(y); out = {}
    for k, p in preds.items():
        C = np.zeros((G, K * K)); np.add.at(C, (codes, y * K + np.asarray(p)), 1.0)
        out[k] = (np.array(confusion_metrics(C.sum(0).reshape(K, K))), np.stack(confusion_metrics((cnt @ C).reshape(n, K, K)), 1))
    return out

def ci(a, axis=0):
    return np.percentile(a, [2.5, 97.5], axis=axis)

def fmt_ci(x, lo, hi, d=3):
    return f"{x:.{d}f} ({lo:.{d}f}, {hi:.{d}f})"

def save(df, name, **kw):
    df.to_csv(OUT / name, **kw); print(f"written: outputs/{name}")

print("ROOT:", ROOT); print("OUT :", OUT)

Mounted at /content/drive
ROOT: /content/drive/MyDrive/NLP_Projects/03_DrugReview
OUT : /content/drive/MyDrive/NLP_Projects/03_DrugReview/08_Manuscript_evaluation/outputs


## 1. Rungs 1 and 2: the binned rating and the holistic label

The seven model families, each scored against its own label, with text only and with the review's condition and ingredients added. The paired difference isolates the metadata: the same model, the same reviews, the same resamples.

In [2]:
# Rungs 1 and 2 are scored inside their own notebooks (01/02 and the _aug variants); their narrative-cluster summaries are read here.
FOLDERS = [("01_OriginalLabel", "binned rating", "text"), ("02_4ominiLabel", "holistic label", "text"),
           ("01_OriginalLabel_aug", "binned rating", "text + metadata"), ("02_4ominiLabel_aug", "holistic label", "text + metadata")]
MODEL_NAMES = {"lr": "LR", "rf": "RF", "lgbm": "LightGBM", "gru": "GRU", "cnn": "CNN", "albert": "ALBERT", "biobert": "BioBERT"}
rows = []
for folder, target, inp in FOLDERS:
    r = pd.read_csv(ROOT / folder / "outputs" / "reported_metrics_bootstrap.csv")
    for _, x in r.iterrows():
        rows.append({"target": target, "input": inp, "model": MODEL_NAMES[x.model], "precision_w": round(x.precision_w, 3), "recall_w": round(x.recall_w, 3),
                     "weighted_f1": fmt_ci(x.f1_weighted, x.f1_weighted_lo, x.f1_weighted_hi), "macro_auc": fmt_ci(x.auc_macro, x.auc_macro_lo, x.auc_macro_hi),
                     "mae": fmt_ci(x.mae, x.mae_lo, x.mae_hi), "qwk": fmt_ci(x.qwk, x.qwk_lo, x.qwk_hi)})
t12 = pd.DataFrame(rows)
for (target, inp), g in t12.groupby(["input", "target"], sort=False):
    print(f"\n{target} | {inp}\n" + g.drop(columns=["target", "input"]).to_string(index=False))
save(t12, "rung12_metrics.csv", index=False)

# Metadata: paired difference in weighted F1, text + metadata minus text, for each model and label (narrative-group bootstrap, 1,000 draws, seed 2025).
split = pd.read_csv(ROOT / "00_split" / "data" / "DrugReview_split.csv", usecols=["uniqueID", "narrative_group", "split"])
def rung12_predictions(folder):
    ml = pd.read_csv(ROOT / folder / "outputs" / "ml_models_predictions.csv", usecols=["uniqueID", "true_label_id", "lr_pred", "rf_pred", "lgbm_pred"])
    dl = pd.read_csv(ROOT / folder / "outputs" / "dl_models_predictions.csv", usecols=["uniqueID", "true_label_id"] + [f"{m}_pred_id" for m in ("gru", "cnn", "albert", "biobert")])
    for m in ("lr", "rf", "lgbm"):
        ml[f"{m}_pred_id"] = ml[f"{m}_pred"].map({c: i for i, c in enumerate(CLASSES)})
    d = ml.drop(columns=["lr_pred", "rf_pred", "lgbm_pred"]).merge(dl, on=["uniqueID", "true_label_id"], validate="one_to_one")
    assert d.filter(like="_pred_id").notna().all().all(), f"unmapped predictions in {folder}"
    return d.merge(split, on="uniqueID", validate="one_to_one").sort_values("uniqueID").reset_index(drop=True)
rows = []
for base, aug, target in (("01_OriginalLabel", "01_OriginalLabel_aug", "binned rating"), ("02_4ominiLabel", "02_4ominiLabel_aug", "holistic label")):
    t, a = rung12_predictions(base), rung12_predictions(aug)
    assert (t.uniqueID.values == a.uniqueID.values).all() and (t.true_label_id.values == a.true_label_id.values).all() and (t.split == "test").all()
    preds = {**{f"text|{m}": t[f"{m}_pred_id"].values for m in MODEL_NAMES}, **{f"meta|{m}": a[f"{m}_pred_id"].values for m in MODEL_NAMES}}
    bs = boot_metrics(t.true_label_id.values, preds, t.narrative_group.values, 5, 1000, 2025)
    for m, name in MODEL_NAMES.items():
        dpt = bs[f"meta|{m}"][0][0] - bs[f"text|{m}"][0][0]; lo, hi = ci(bs[f"meta|{m}"][1][:, 0] - bs[f"text|{m}"][1][:, 0])
        rows.append({"target": target, "model": name, "wf1_text": round(bs[f"text|{m}"][0][0], 3), "wf1_text_plus_metadata": round(bs[f"meta|{m}"][0][0], 3),
                     "delta": round(dpt, 3), "delta_lo": round(lo, 3), "delta_hi": round(hi, 3)})
meta = pd.DataFrame(rows); print("\nMetadata effect: weighted F1, text + metadata minus text\n" + meta.to_string(index=False))
print(f"\nlargest absolute change: {meta.delta.abs().max():.3f}; transformers: {meta[meta.model.isin(['ALBERT', 'BioBERT'])][['target', 'model', 'delta', 'delta_lo', 'delta_hi']].to_string(index=False)}")
save(meta, "rung12_metadata_paired.csv", index=False)

# The ALBERT - BioBERT difference within each label comes from the paired narrative-cluster intervals saved by the rung 1-2 notebooks.
rows = []
for folder, target in (("01_OriginalLabel", "binned rating"), ("02_4ominiLabel", "holistic label")):
    pdiff = pd.read_csv(ROOT / folder / "outputs" / "paired_differences_bootstrap.csv")
    r = pdiff[(pdiff.model_1 == "albert") & (pdiff.model_2 == "biobert") & (pdiff.metric == "f1_weighted")].iloc[0]
    rows.append({"target": target, "contrast": "ALBERT - BioBERT, weighted F1", "difference": fmt_ci(r["diff"], r.diff_lo, r.diff_hi)})
print("\n" + pd.DataFrame(rows).to_string(index=False))


text | binned rating
   model  precision_w  recall_w          weighted_f1            macro_auc                  mae                  qwk
      LR        0.526     0.590 0.537 (0.529, 0.545) 0.784 (0.780, 0.789) 0.759 (0.742, 0.775) 0.611 (0.599, 0.622)
      RF        0.505     0.545 0.517 (0.509, 0.525) 0.739 (0.734, 0.744) 0.875 (0.857, 0.893) 0.546 (0.533, 0.559)
LightGBM        0.541     0.531 0.535 (0.527, 0.542) 0.757 (0.752, 0.763) 0.806 (0.790, 0.822) 0.611 (0.599, 0.622)
     GRU        0.573     0.562 0.567 (0.559, 0.574) 0.799 (0.794, 0.803) 0.681 (0.667, 0.696) 0.695 (0.684, 0.705)
     CNN        0.568     0.556 0.558 (0.552, 0.566) 0.795 (0.791, 0.799) 0.725 (0.709, 0.740) 0.664 (0.654, 0.675)
  ALBERT        0.640     0.638 0.638 (0.631, 0.645) 0.862 (0.858, 0.865) 0.493 (0.482, 0.505) 0.815 (0.808, 0.823)
 BioBERT        0.631     0.622 0.625 (0.618, 0.633) 0.853 (0.849, 0.857) 0.525 (0.513, 0.537) 0.799 (0.791, 0.807)

text | holistic label
   model  precision_w  reca

## 2. Rung 3: the aspect baseline

Four three-class aspect heads and no holistic head. This is the reference for every aspect comparison of rungs 4–7.

In [3]:
# Rung 3: four three-class aspect heads, test partition; weighted F1 with narrative-group intervals (1,000 draws, seed 2025), macro F1, per-class F1.
rows = []
for enc in ENCODERS:
    t = test_predictions("03", enc)
    for a in ASPECTS:
        y, p = t[f"{a}_true_id"].values, t[f"{a}_pred_id"].values
        (pt, bt), = boot_metrics(y, {"m": p}, t.narrative_group.values, 3, 1000, 2025).values(); lo, hi = ci(bt[:, 0])
        per = f1_score(y, p, labels=[0, 1, 2], average=None)
        rows.append({"aspect": a, "encoder": enc, "weighted_f1": fmt_ci(pt[0], lo, hi), "macro_f1": round(pt[1], 3),
                     "f1_negative": round(per[0], 3), "f1_neutral": round(per[1], 3), "f1_positive": round(per[2], 3)})
t3 = pd.DataFrame(rows).sort_values(["aspect", "encoder"], key=lambda s: s.map({a: i for i, a in enumerate(ASPECTS)}) if s.name == "aspect" else s)
print(t3.to_string(index=False)); save(t3, "rung3_aspect_baseline.csv", index=False)

  aspect encoder          weighted_f1  macro_f1  f1_negative  f1_neutral  f1_positive
efficacy  albert 0.788 (0.782, 0.794)     0.713        0.768       0.496        0.874
efficacy biobert 0.783 (0.778, 0.789)     0.708        0.758       0.497        0.869
  safety  albert 0.798 (0.792, 0.803)     0.785        0.842       0.747        0.765
  safety biobert 0.798 (0.792, 0.804)     0.786        0.842       0.749        0.766
  burden  albert 0.695 (0.688, 0.701)     0.634        0.716       0.723        0.463
  burden biobert 0.697 (0.690, 0.703)     0.639        0.723       0.719        0.475
    cost  albert 0.974 (0.972, 0.976)     0.854        0.797       0.988        0.776
    cost biobert 0.973 (0.971, 0.976)     0.848        0.790       0.987        0.768
written: outputs/rung3_aspect_baseline.csv


## 3. Rungs 4–7: holistic and aspect scores

Every configuration predicts the holistic label; the aspect columns score its own aspect heads.

In [4]:
# Rungs 4-7: holistic label (weighted F1 with narrative-group intervals, 1,000 draws, seed 2025; macro F1; MAE and QWK on codes 0-4)
# and each configuration's own aspect heads (three-class weighted F1; --- where the configuration has none).
ORDER = ["04", "A0", "5p", "5r", "5j", "5d", "6ff", "6a", "6f", "7e", "7c"]
rows = []
for run in ORDER:
    for enc in ENCODERS:
        t = test_predictions(run, enc); y, p = t.overall_true_id.values, t.overall_pred_id.values
        (pt, bt), = boot_metrics(y, {"m": p}, t.narrative_group.values, 5, 1000, 2025).values(); lo, hi = ci(bt[:, 0])
        r = {"configuration": NAMES[run], "encoder": enc, "weighted_f1": fmt_ci(pt[0], lo, hi), "macro_f1": round(pt[1], 3),
             "mae": round(float(np.abs(y - p).mean()), 3), "qwk": round(pt[2], 3)}
        for a in ASPECTS:
            has = f"{a}_pred_id" in t and t[f"{a}_pred_id"].notna().all()
            r[f"{a}_wf1"] = round(f1_score(t[f"{a}_true_id"], t[f"{a}_pred_id"], average="weighted"), 3) if has else "---"
        rows.append(r)
t7 = pd.DataFrame(rows); print(t7.to_string(index=False)); save(t7, "rungs4to7_scores.csv", index=False)

         configuration encoder          weighted_f1  macro_f1   mae   qwk efficacy_wf1 safety_wf1 burden_wf1 cost_wf1
            4 parallel  albert 0.788 (0.782, 0.793)     0.712 0.267 0.915        0.764      0.748      0.647    0.969
            4 parallel biobert 0.782 (0.775, 0.788)     0.695 0.275 0.910        0.773      0.779      0.665    0.967
     4 no aspect heads  albert 0.785 (0.779, 0.791)     0.708 0.268 0.916          ---        ---        ---      ---
     4 no aspect heads biobert 0.776 (0.770, 0.782)     0.694 0.284 0.907          ---        ---        ---      ---
              5 fusion  albert 0.790 (0.784, 0.795)     0.712 0.258 0.921        0.781      0.785      0.661     0.97
              5 fusion biobert 0.778 (0.771, 0.784)     0.694 0.282 0.908        0.777      0.786      0.666    0.969
            5 residual  albert 0.788 (0.782, 0.793)     0.709 0.264 0.917        0.784      0.784      0.679    0.971
            5 residual biobert 0.783 (0.777, 0.789)     

## 4. Paired holistic contrasts and the equivalence margin

First-named configuration minus second-named, on the same test reviews and the same 10,000 draws. The margin was fixed at ±0.01 before the comparisons were run; the verdicts are also shown under narrower and wider margins and with the first 1,000 draws.

In [5]:
# Paired holistic contrasts: 10,000 narrative-group draws, seed 2025 (the first 1,000 draws are the earlier 1,000-draw analysis).
# The equivalence verdict asks whether the unrounded 95% interval of the weighted-F1 difference lies inside the margin;
# "borderline" marks an interval endpoint within 0.001 of +/-0.01.
N_DRAWS, SEED, MARGINS, K = 10000, 2025, (0.005, 0.01, 0.015, 0.02), 5
CONTRASTS = [("4 parallel - no aspect heads", "04", "A0"), ("5 residual - fusion", "5r", "5p"), ("5 bottleneck, joint - fusion", "5j", "5p"),
             ("5 bottleneck, detached - fusion", "5d", "5p"), ("6 (4+0) - (4+4)", "6a", "6ff"), ("6 (0+4) - (4+4)", "6f", "6ff"),
             ("7 evidence loss off - on", "7c", "7e"), ("5 fusion - 4 parallel", "5p", "04"), ("6 (4+4) - 4 parallel", "6ff", "04"),
             ("7 evidence loss on - 4 parallel", "7e", "04"), ("7 evidence loss on - 5 residual", "7e", "5r"), ("6 (4+0) - 5 fusion", "6a", "5p"),
             ("7 evidence loss off - 4 parallel", "7c", "04"), ("6 (0+4) - (4+0)", "6f", "6a"), ("4 no aspect heads - 2 holistic", "A0", "02")]
rows = []
for enc in ENCODERS:
    P = {k: test_predictions(k, enc, ["uniqueID", "narrative_group", "overall_true_id", "overall_pred_id"]) for k in RUNS if k != "03"}
    d2 = pd.read_csv(ROOT / "02_4ominiLabel" / "outputs" / "dl_models_predictions.csv", usecols=["uniqueID", "true_label_id", f"{enc}_pred_id"]).sort_values("uniqueID").reset_index(drop=True)
    base = P["04"]; y = base.overall_true_id.values
    assert all((p.uniqueID.values == base.uniqueID.values).all() and (p.overall_true_id.values == y).all() for p in P.values())
    assert (d2.uniqueID.values == base.uniqueID.values).all() and (d2.true_label_id.values == y).all()
    preds = {k: v.overall_pred_id.values for k, v in P.items()}; preds["02"] = d2[f"{enc}_pred_id"].values
    bs = boot_metrics(y, preds, base.narrative_group.values, K, N_DRAWS, SEED)
    for name, a, b in CONTRASTS:
        r = {"contrast": name, "encoder": enc}
        for mi, m in enumerate(("wf1", "mf1", "qwk")):
            dpt = bs[a][0][mi] - bs[b][0][mi]; bt = bs[a][1][:, mi] - bs[b][1][:, mi]; lo, hi = ci(bt); lo1k, hi1k = ci(bt[:1000])
            r.update({f"{m}_delta": dpt, f"{m}_lo": lo, f"{m}_hi": hi, f"{m}_lo_1k": lo1k, f"{m}_hi_1k": hi1k})
            if m == "wf1":
                sd = ci(bt.reshape(10, 1000), axis=1); r["wf1_endpoint_sd_1k"] = max(sd[0].std(ddof=1), sd[1].std(ddof=1))
        for mg in MARGINS:
            r[f"inside_{mg}"] = "yes" if (r["wf1_lo"] > -mg and r["wf1_hi"] < mg) else "no"
        r["inside_0.01_1k"] = "yes" if (r["wf1_lo_1k"] > -0.01 and r["wf1_hi_1k"] < 0.01) else "no"
        r["borderline_0.01"] = min(abs(abs(r["wf1_lo"]) - 0.01), abs(abs(r["wf1_hi"]) - 0.01)) < 0.001
        rows.append(r)
c = pd.DataFrame(rows)
show = c.assign(wf1=[fmt_ci(a, b, d) for a, b, d in zip(c.wf1_delta, c.wf1_lo, c.wf1_hi)], qwk=[fmt_ci(a, b, d) for a, b, d in zip(c.qwk_delta, c.qwk_lo, c.qwk_hi)],
                wf1_4dp=[fmt_ci(a, b, d, 4) for a, b, d in zip(c.wf1_delta, c.wf1_lo, c.wf1_hi)])
print(show[["contrast", "encoder", "wf1", "qwk", "inside_0.01", "borderline_0.01"]].to_string(index=False))
print("\nMargin sensitivity (10,000 draws; last column 1,000 draws)\n" + show[["contrast", "encoder", "wf1_4dp"] + [f"inside_{m}" for m in MARGINS] + ["inside_0.01_1k"]].to_string(index=False))
print(f"\n{len(c)} contrasts; inside the margin: " + ", ".join(f"+/-{m}: {(c[f'inside_{m}'] == 'yes').sum()}" for m in MARGINS)
      + f"; +/-0.01 with 1,000 draws: {(c['inside_0.01_1k'] == 'yes').sum()}; borderline at +/-0.01: {int(c['borderline_0.01'].sum())}; "
      + f"largest SD of a 1,000-draw endpoint: {c.wf1_endpoint_sd_1k.max():.4f}")
save(c, "contrasts_10k.csv", index=False)

                        contrast encoder                     wf1                     qwk inside_0.01  borderline_0.01
    4 parallel - no aspect heads  albert   0.003 (-0.003, 0.008)  -0.001 (-0.005, 0.002)         yes            False
             5 residual - fusion  albert  -0.002 (-0.007, 0.003) -0.004 (-0.007, -0.001)         yes            False
    5 bottleneck, joint - fusion  albert -0.017 (-0.023, -0.012) -0.009 (-0.012, -0.006)          no            False
 5 bottleneck, detached - fusion  albert -0.155 (-0.162, -0.147) -0.079 (-0.084, -0.075)          no            False
                 6 (4+0) - (4+4)  albert    0.006 (0.001, 0.011)   0.001 (-0.002, 0.004)          no             True
                 6 (0+4) - (4+4)  albert -0.007 (-0.012, -0.002) -0.004 (-0.007, -0.001)          no            False
        7 evidence loss off - on  albert   0.000 (-0.004, 0.005)  -0.002 (-0.005, 0.001)         yes            False
           5 fusion - 4 parallel  albert   0.002 (-0.003

## 5. Aspect agreement relative to rung 3

Each configuration's aspect head minus the rung-3 head of the same encoder, paired on the same reviews and draws.

In [6]:
# Aspect agreement relative to rung 3: each configuration's aspect head minus the rung-3 head of the same encoder, three-class
# weighted F1, paired narrative-group bootstrap (1,000 draws, seed 2025).
ORDER = ["04", "5p", "5r", "5j", "5d", "6ff", "6a", "7e", "7c"]
rows = []
for enc in ENCODERS:
    base = test_predictions("03", enc)
    for run in ORDER:
        t = test_predictions(run, enc); assert (t.uniqueID.values == base.uniqueID.values).all()
        r = {"configuration": NAMES[run], "encoder": enc}
        for a in ASPECTS:
            assert (t[f"{a}_true_id"].values == base[f"{a}_true_id"].values).all()
            bs = boot_metrics(base[f"{a}_true_id"].values, {"run": t[f"{a}_pred_id"].values, "r3": base[f"{a}_pred_id"].values}, base.narrative_group.values, 3, 1000, 2025)
            dpt = bs["run"][0][0] - bs["r3"][0][0]; lo, hi = ci(bs["run"][1][:, 0] - bs["r3"][1][:, 0])
            r[a] = f"{dpt:+.3f} ({lo:+.3f}, {hi:+.3f})"
        rows.append(r)
t9 = pd.DataFrame(rows); print(t9.to_string(index=False)); save(t9, "aspect_delta_vs_rung3.csv", index=False)

         configuration encoder                efficacy                  safety                  burden                    cost
            4 parallel  albert -0.024 (-0.029, -0.019) -0.050 (-0.055, -0.043) -0.048 (-0.054, -0.041) -0.006 (-0.007, -0.004)
              5 fusion  albert -0.007 (-0.013, -0.001) -0.012 (-0.018, -0.007) -0.034 (-0.040, -0.028) -0.004 (-0.006, -0.002)
            5 residual  albert -0.004 (-0.009, +0.001) -0.014 (-0.019, -0.008) -0.015 (-0.022, -0.009) -0.004 (-0.005, -0.002)
   5 bottleneck, joint  albert -0.041 (-0.047, -0.034) -0.044 (-0.050, -0.038) -0.143 (-0.151, -0.136) -0.011 (-0.012, -0.009)
5 bottleneck, detached  albert -0.010 (-0.015, -0.005) -0.009 (-0.014, -0.003) -0.028 (-0.034, -0.023) -0.007 (-0.009, -0.005)
 6 4 anchored + 4 free  albert -0.016 (-0.021, -0.010) -0.027 (-0.033, -0.022) -0.027 (-0.034, -0.022) -0.005 (-0.007, -0.003)
 6 4 anchored + 0 free  albert -0.015 (-0.020, -0.009) -0.014 (-0.019, -0.009) -0.015 (-0.021, -0.009) -0.004 (

## 6. Checkpoint selection

Rungs 4–7 keep the epoch with the best validation holistic F1. This section asks how much aspect agreement a selection on mean validation aspect F1 could have gained, using the saved training histories only.

In [7]:
# Checkpoint selection: aspect agreement each run could reach if its checkpoint were chosen on mean validation aspect F1 instead of
# holistic F1 (validation histories only; the test partition is not used).
R = {"3 aspect baseline": "03", "4 parallel": "04", "5 fusion": "5p", "5 residual": "5r", "6 4 anchored + 0 free": "6a", "7 evidence loss on": "7e", "7 evidence loss off": "7c"}
rows = []
for enc in ENCODERS:
    for name, k in R.items():
        path = ROOT / RUNS[k] / enc; sel = json.load(open(path / "selected_parameters.json")); recs = []
        for f in sorted(glob.glob(str(path / "trial_*_history.json"))):
            trial = int(Path(f).name.split("_")[1])
            for row in json.load(open(f)):
                v = row["validation"]
                recs.append({"trial": trial, "epoch": row["epoch"], "aspect_mean": float(np.mean([v[a]["weighted_f1"] for a in ASPECTS])),
                             "holistic": v["overall"]["weighted_f1"] if "overall" in v else np.nan})
        h = pd.DataFrame(recs); chosen = h[(h.trial == sel["trial"]) & (h.epoch == sel["selected_epoch"])].iloc[0]; best = h.loc[h.aspect_mean.idxmax()]
        rows.append({"encoder": enc, "run": name, "selected_trial_epoch": f"{sel['trial']}/{sel['selected_epoch']}",
                     "holistic_at_selected": round(chosen.holistic, 3) if not np.isnan(chosen.holistic) else "---", "aspect_mean_at_selected": round(chosen.aspect_mean, 3),
                     "best_aspect_mean": round(best.aspect_mean, 3), "holistic_at_best_aspect": round(best.holistic, 3) if not np.isnan(best.holistic) else "---",
                     "gain_if_selected_on_aspects": round(best.aspect_mean - chosen.aspect_mean, 3), "checkpoints": len(h)})
s11 = pd.DataFrame(rows); print(s11.to_string(index=False)); print(f"\nlargest possible gain in mean aspect F1: {s11.gain_if_selected_on_aspects.max():.3f}")
save(s11, "checkpoint_selection.csv", index=False)

encoder                   run selected_trial_epoch holistic_at_selected  aspect_mean_at_selected  best_aspect_mean holistic_at_best_aspect  gain_if_selected_on_aspects  checkpoints
 albert     3 aspect baseline                 10/8                  ---                    0.814             0.814                     ---                        0.000           70
 albert            4 parallel                  9/6                0.783                    0.781             0.786                   0.775                        0.005           71
 albert              5 fusion                 10/6                0.782                    0.798             0.803                   0.781                        0.006           71
 albert            5 residual                 10/7                0.785                    0.805             0.807                   0.785                        0.002           71
 albert 6 4 anchored + 0 free                 10/7                0.786                    0.80

## 7. Fusion weights

The review-specific weights $\beta$ with which the holistic head fuses the aspect pools, averaged over the test partition, overall and by the review's holistic label. The per-class means are checked against the tables the rung notebooks saved.

In [8]:
# Fusion weights beta: mean over the test partition, overall and by the review's holistic label, for every configuration that fuses aspect pools.
rows = []
for run in ("5p", "5r", "7e", "7c"):
    for enc in ENCODERS:
        t = test_predictions(run, enc); w = [f"fusion_w_{a}" for a in ASPECTS]
        allr = t[w].mean(); rows.append({"configuration": NAMES[run], "encoder": enc, "holistic_class": "all reviews", **{a: round(allr[f"fusion_w_{a}"], 3) for a in ASPECTS}})
        byc = t.groupby("overall_true_id")[w].mean()
        for i, c in enumerate(CLASSES):
            rows.append({"configuration": NAMES[run], "encoder": enc, "holistic_class": c, **{a: round(byc.loc[i, f"fusion_w_{a}"], 3) for a in ASPECTS}})
        saved = pd.read_csv(ROOT / RUNS[run] / enc / f"{enc}_fusion_by_holistic_class.csv", index_col=0)
        assert np.allclose(saved.reindex(CLASSES).values, byc.values, atol=5e-4), f"{run} {enc}: saved fusion table differs"
s13 = pd.DataFrame(rows); print(s13.to_string(index=False)); save(s13, "fusion_weights.csv", index=False)

      configuration encoder holistic_class  efficacy  safety  burden  cost
           5 fusion  albert    all reviews     0.395   0.164   0.140 0.302
           5 fusion  albert  VERY_NEGATIVE     0.270   0.296   0.180 0.254
           5 fusion  albert       NEGATIVE     0.299   0.256   0.176 0.269
           5 fusion  albert        NEUTRAL     0.341   0.197   0.181 0.281
           5 fusion  albert       POSITIVE     0.476   0.095   0.110 0.320
           5 fusion  albert  VERY_POSITIVE     0.446   0.101   0.121 0.332
           5 fusion biobert    all reviews     0.295   0.246   0.219 0.240
           5 fusion biobert  VERY_NEGATIVE     0.262   0.266   0.235 0.236
           5 fusion biobert       NEGATIVE     0.279   0.259   0.223 0.240
           5 fusion biobert        NEUTRAL     0.295   0.248   0.218 0.239
           5 fusion biobert       POSITIVE     0.313   0.235   0.212 0.241
           5 fusion biobert  VERY_POSITIVE     0.305   0.239   0.214 0.242
         5 residual  albe